In [0]:
pip install pytz  # Si no está en el cluster

In [0]:
# Intento de lectura de los secretos guardados
try:
    user = dbutils.secrets.get(scope="telemetria_patio", key="USERNAME")
    base_url = dbutils.secrets.get(scope="telemetria_patio", key="BASE_URL")
    
    print("✅ Conexión exitosa con el Scope 'telemetria_patio'")
    print(f"Usuario recuperado: {user}")
    print(f"URL recuperada: {base_url}")
    print("-" * 30)
    print("Nota: El valor de PASSWORD no se puede imprimir por seguridad (saldrá como [REDACTED])")
    
except Exception as e:
    print("❌ Error al recuperar los secretos. Revisa el nombre del Scope o las Keys.")
    print(f"Detalle: {e}")

In [0]:
# MAGIC %pip install pytz

import requests
import json
from datetime import datetime, timedelta
import pandas as pd

# ======================================================
# 1. RECUPERAR CREDENCIALES
# ======================================================
SCOPE = "telemetria_patio"
try:
    BASE_URL = dbutils.secrets.get(scope=SCOPE, key="BASE_URL")
    USERNAME = dbutils.secrets.get(scope=SCOPE, key="USERNAME")
    PASSWORD = dbutils.secrets.get(scope=SCOPE, key="PASSWORD")
    ASSET_ID = dbutils.secrets.get(scope=SCOPE, key="ASSET_ID")
    print("✅ Credenciales recuperadas.")
except Exception as e:
    raise Exception(f"❌ Error de Seguridad: {e}")

# ======================================================
# 2. EXTRACCIÓN (EXTRACT)
# ======================================================
ahora = datetime.now()
end_ts = int(ahora.timestamp() * 1000)
start_ts = int((ahora - timedelta(days=1)).timestamp() * 1000)

session = requests.Session()
login_resp = session.post(f"{BASE_URL}/api/auth/login", json={"username": USERNAME, "password": PASSWORD})
token = login_resp.json()["token"]
session.headers.update({"X-Authorization": f"Bearer {token}", "Content-Type": "application/json"})

KEYS = "logs_nia,logs_ubicacion,shared_tipo,shared_placaTracto,shared_placaPlataforma,shared_tracker,shared_conductor,shared_empresa"
url = f"{BASE_URL}/api/plugins/telemetry/ASSET/{ASSET_ID}/values/timeseries?keys={KEYS}&startTs={start_ts}&endTs={end_ts}&agg=NONE&order=ASC"
response = session.get(url)
raw_payload = response.json()
print("✅ Datos extraídos de la API.")

# ======================================================
# 3. CARGA DIRECTA A DELTA (EVITANDO EL DISCO LOCAL)
# ======================================================
try:
    # Convertimos el JSON de la API a una lista de diccionarios plana
    # La API devuelve un dict donde cada llave es una métrica con una lista de valores
    rows = []
    for key, values in raw_payload.items():
        for item in values:
            rows.append({
                "metrica": key,
                "ts": item["ts"],
                "valor": str(item["value"]) # Lo guardamos como string para la capa Bronze
            })

    # Creamos un DataFrame de Pandas (en memoria)
    pdf = pd.DataFrame(rows)
    
    # Lo convertimos a Spark DataFrame y lo persistimos como Tabla Delta (Capa Bronze)
    df_spark = spark.createDataFrame(pdf)
    
    # Nombre de la tabla para tu proyecto
    nombre_tabla = "telemetria_patio_bronze"
    
    df_spark.write.format("delta").mode("append").saveAsTable(nombre_tabla)
    
    print(f"📦 PROCESO FINALIZADO.")
    print(f"Datos persistidos en la tabla: {nombre_tabla}")
    display(spark.table(nombre_tabla).limit(5))

except Exception as e:
    print(f"❌ Error al persistir en Delta: {e}")